In [56]:
import numpy as np

import mbuild as mb
from mbuild import Polymer
from gmso.core.forcefield import ForceField

gaff = ForceField("../files/gaff.xml")
oplsa = ForceField("oplsaa")

# Building polymers in mBuild
- Tagging bonding sites with SMILES strings:
    - This implementation is inspired by polymer syntax approaches such as BIGSmiles, CGSmiles and more.
    - It is not meant to be a full implementation of the above syntax approaches
    - You can put any tag value inside the `{}` brackets and carry it over to the polymer builder

In [57]:
monomer = mb.load("c1c{<}sc{>}c1", smiles=True)
chain = Polymer()
chain.add_monomer(monomer, head_tag=">", tail_tag="<", separation=0.145)
chain.build(n=4)
chain.visualize().show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [58]:
peg_monomer = mb.load("C{<}CO{>}", smiles=True)
peg_chain = Polymer()
peg_chain.add_monomer(peg_monomer, head_tag=">", tail_tag="<", separation=0.145)
peg_chain.build(n=34)
peg_chain.visualize()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## Correct bonding graph, but bad configurations
- By using tagged SMILES strings, and the Polymer() class in mBuild, we can easily generate the correct polymer bonding topology
- However, we have no control over the chain configuraiton
- **Try:** Change `n` to a large number (40 - 50) in the thiophene example and look at the resulting configuration.
- **Challenges:** Long straight chains create challenges in building many polymer systems

## The `Path` module:
- Designed to enable programmatic control over polymer configurations.
- Extensible: You can make your own `Path` functions if the built-in ones don't cover your needs
- **Note:** In these functions, the `spacing` parameter is the center-to-center distance

In [89]:
from mbuild.path import Path, straight_line, lamellar, cyclic, hard_sphere_random_walk, lamellar, knot

In [90]:
line_path = straight_line(N=20, spacing=0.40)
line_path.visualize(radius=0.40)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [91]:
lamellar_2D = lamellar(
    spacing=0.45,
    layer_length=5,
    layer_separation=1.2,
    num_layers=5
)
lamellar_2D.visualize(radius=0.40)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [92]:
lamellar_3D = lamellar(
    spacing=0.45,
    layer_length=5,
    layer_separation=1.2,
    num_layers=5,
    stack_separation=1.8,
    num_stacks=4
)
lamellar_3D.visualize(radius=0.40)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [93]:
ring = cyclic(N=30, spacing=0.45)
ring.visualize(radius=0.45)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

# Using `Path` with mBuild's Polymer builder

- Paths are essentially coarse-grained polymer configurations.
- These can be converted to mBuild Compounds, and used as-is within the MoSDeF workflow.
    - `path_compound = path.to_compound()`
    - Make and use your own GMSO xml with bead definitions and parameters (works with arbitrary beads as well!)
- These can also be used by mBuild's `Polymer` builder to set chain configurations for atomistic polymers.
    - Above: We used `Polymer.build(N)`
    - Here: We can use `Polymer.build_from_path(path)`

In [94]:
# Create an mbuild.Path instance
ring = cyclic(N=30, spacing=0.30)

# Follow the same steps to build a Polymer; call a different build-method
peg_monomer = mb.load("C{<}CO{>}", smiles=True)
peg_chain = Polymer()
peg_chain.add_monomer(peg_monomer, head_tag=">", tail_tag="<", separation=0.145)

# NOTE: For a ring polymer example we need to manually set 2 parameters
# bond_head_tail = True --> Completes the head-tail bond to form a closed ring
# add_hydrogens = False --> Prevents hydrogen capping of the head and tail monomers

peg_chain.build_from_path(path=ring, bond_head_tail=True, add_hydrogens=False)
peg_chain.visualize()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [95]:
lamellar_2D = lamellar(spacing=0.45, layer_length=5, layer_separation=1.2, num_layers=5)
monomer = mb.load("c1c{<}sc{>}c1", smiles=True)
chain = Polymer()
chain.add_monomer(monomer, head_tag=">", tail_tag="<", separation=0.145)
chain.build_from_path(lamellar_2D)
chain.visualize()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [96]:
lamellar_3D = lamellar(spacing=0.40, layer_length=5, layer_separation=1.0, num_layers=5, stack_separation=1.0, num_stacks=4)
monomer = mb.load("c1c{<}sc{>}c1", smiles=True)
chain = Polymer()
chain.add_monomer(monomer, head_tag=">", tail_tag="<", separation=0.145)
chain.build_from_path(lamellar_3D)
chain.visualize()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [97]:
random_walk_path = hard_sphere_random_walk(radius=0.30, bond_length=0.301, termination=25)
random_walk_path.visualize(radius=0.30)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [98]:
peg_monomer = mb.load("C{<}CO{>}", smiles=True)
peg_chain = Polymer()
peg_chain.add_monomer(peg_monomer, head_tag=">", tail_tag="<", separation=0.145)
peg_chain.build_from_path(path=random_walk_path)
peg_chain.visualize()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

# Back mapping and Energy Minimization
- Back mapping can result in strained bonds, angles and dihedrals, and overlapping particles.

## `mBuild.simulation`
- We interface with multiple simulation methods, including HOOMD-Blue, OpenMM, and OpenBabel
- **HOOMD-Blue:** Python API (no input files), usable within MoSDeF workflow with gmso and foyer forcefields.
    - Python API allows running simulations in for loops, while loops, conditionals, etc.
    - Update mBuild Compound coordinates in place
    - Run on GPUs for large mBuild Compounds (multi-component systems)
- `mBuild.simulation` proves a wrapper over HOOMD's API, for usage within mBuild, operating directly on an mBuild Compound
    -  `HoomdSimulation`: Simulation context management. Created once, used by multiple simulation methods
    -  `ForcesHandler`: Modulate the forcefield used for relaxation. Remove components for faster performance, switch between LJ and DPD pair forces, and/or scale force constants.
    -  `hoomd_fire`, `hoomd_cap_displacement`, `hoomd_nvt`: Simulation methods that can be called in series, within for loops, while loops, etc.

In [99]:
from mbuild.simulation import HoomdSimulation, hoomd_fire, hoomd_cap_displacement, hoomd_nvt, ForcesHandler

from gmso.core.forcefield import ForceField
from gmso.parameterization.parameterize import apply

gaff = ForceField("../files/gaff.xml")
oplsaa = ForceField("oplsaa")

In [104]:
peg_monomer = mb.load("C{<}CO{>}", smiles=True)
print(peg_monomer.volume()**(1/3))
spacing = peg_monomer.volume()**(1/3) * 0.80

0.3783678846142872


In [105]:
random_walk_path = hard_sphere_random_walk(radius=spacing, bond_length=spacing+0.1, rw_angles=(1.5, 3.14), termination=25)

peg_monomer = mb.load("C{<}CO{>}", smiles=True)
peg_chain = Polymer()
peg_chain.add_monomer(peg_monomer, head_tag=">", tail_tag="<", separation=0.145)
peg_chain.build_from_path(path=random_walk_path)
peg_chain.visualize()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

### Relaxing: `hoomd_cap_displacement` + `hoomd_fire`

In [106]:
hoomd_sim = HoomdSimulation(compound=peg_chain, forcefield=gaff, r_cut=1.2)
hoomd_cap_displacement(
    n_steps=1000,
    compound=peg_chain,
    sim=hoomd_sim,
    forces_handler=ForcesHandler(),
    max_displacement=1e-3,
    dt=1
)

hoomd_fire(compound=peg_chain, n_iterations=5, n_steps=1000, sim=hoomd_sim, forces_handler=ForcesHandler())
peg_chain.visualize()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

#### Result:
- The cell below shows the CG path and the relaxed atomistic polymer in a single mBuild Compound
- We can see that the chain configuration is mostly preserved

In [107]:
example = mb.clone(peg_chain)
example.add(random_walk_path.to_compound())
example.visualize(bead_size=0.5)

2026-07-01 12:20:18,370 - mbuild.conversion - WARNING - No element attribute associated with '<_A pos=([ 0.1658 -0.037   0.217 ]), 1 bonds, id: 13302230976>'; and no matching elements found based upon the compound name. Setting atomic number to zero.


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

### hoomd_nvt
- If we run an NVT simulation, the chain configuration changes dramatically (as you might expect)

In [108]:
hoomd_nvt(compound=peg_chain, kT=4, n_steps=50_000, sim=hoomd_sim, forces_handler=ForcesHandler(), dt=1e-4, tau=1e-2)

example = mb.clone(peg_chain)
example.add(random_walk_path.to_compound())
example.visualize(bead_size=0.5)

2026-07-01 12:20:28,599 - mbuild.conversion - WARNING - No element attribute associated with '<_A pos=([ 0.1658 -0.037   0.217 ]), 1 bonds, id: 14103013680>'; and no matching elements found based upon the compound name. Setting atomic number to zero.


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

<h1 style="color: green;">Exercise</h1>

- Build a PEG chain from a path (random walk, lamellar 2D or 3D, cyclic, knot, zigzag)
- Design and run your own energy minimization protocol. Choose between the GAFF and OPLSAA forcefields

Tip: Use the `help` function on the imports below to see their parameters and doc strings.

e.g., `help(knot)`


In [109]:
from mbuild.path import (
    lamellar,
    cyclic,
    straight_line,
    knot,
    hard_sphere_random_walk,
    zigzag,
)

gaff = ForceField("../files/gaff.xml")
oplsa = ForceField("oplsaa")

In [110]:
# build instance of a path
# Call path.visualize(radius= ) to see what the path looks like before passing it to the polymer builder
# Create a PEG monomer, with tagged SMILES indicating the bonding atoms
# Create polymer chain instance
# Use the .add_monomer() method with tagged smiles string for PEG
# Build the chain from the path

# Create a Hoomd simulation object
# Pass the resulting chain into hoomd_cap_displacement, hoomd_fire (you can try hoomd_nvt as well)

---

# Building Copolymers

In [111]:
path = straight_line(N=20, spacing=0.34)

pla_monomer = mb.load("C{<}(C)C(=O)O{>}", smiles=True)
pla_monomer.name = "PLA"
peg_monomer = mb.load("C{<}CO{>}", smiles=True)
peg_monomer.name = "PEG"

copolymer = Polymer()
copolymer.add_monomer(peg_monomer, head_tag=">", tail_tag="<", separation=0.145)
copolymer.add_monomer(pla_monomer, head_tag=">", tail_tag="<", separation=0.145)
copolymer.build_from_path(path=path, sequence="AB")

hoomd_sim = HoomdSimulation(compound=copolymer, forcefield=gaff, r_cut=1.2)
hoomd_cap_displacement(
    n_steps=1000,
    compound=copolymer,
    sim=hoomd_sim,
    forces_handler=ForcesHandler(),
    max_displacement=1e-3,
    dt=1
)

hoomd_fire(compound=copolymer, n_iterations=5, n_steps=1000, sim=hoomd_sim, forces_handler=ForcesHandler())
copolymer.visualize()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [112]:
for child in copolymer.children:
    print(child.name)

PEG
PLA
PEG
PLA
PEG
PLA
PEG
PLA
PEG
PLA
PEG
PLA
PEG
PLA
PEG
PLA
PEG
PLA
PEG
PLA


<h1 style="color: green;">Exercise</h1>

Add a third monomer type to our co-polymer. In this scenario let's use [poly(glycolic acid) (PGA)](https://en.wikipedia.org/wiki/Polyglycolide)

- Build a path object (Choose an N divisible by 3 for this scenario)
- Create a `pga_monomer` Compound with tagged SMILES `pga_monomer = mb.load("O{<}CC{>}(=O)", smiles=True)`

In [113]:
path = ?

pla_monomer = mb.load("C{<}(C)C(=O)O{>}", smiles=True)
peg_monomer = mb.load("C{<}CO{>}", smiles=True)

pga_monomer = ???

copolymer = ?
# Add each of the 3 monomers
# Build with an ABC sequence

# Run energy minimization
hoomd_sim = HoomdSimulation(compound=copolymer, forcefield=gaff, r_cut=1.2)
hoomd_cap_displacement(
    n_steps=1000,
    compound=copolymer,
    sim=hoomd_sim,
    forces_handler=ForcesHandler(),
    max_displacement=1e-3,
    dt=1
)
hoomd_fire(compound=copolymer, n_iterations=5, n_steps=1000, sim=hoomd_sim, forces_handler=ForcesHandler())
copolymer.visualize()

SyntaxError: invalid syntax (1819493457.py, line 1)

<h2 style="color: blue;">Answer</h2>
Click on the cell below to see the answer

In [115]:
path = straight_line(N=21, spacing=0.34)

pla_monomer = mb.load("C{<}(C)C(=O)O{>}", smiles=True)
peg_monomer = mb.load("C{<}CO{>}", smiles=True)

pga_monomer = mb.load("O{<}CC{>}(=O)", smiles=True)

copolymer = Polymer()
copolymer.add_monomer(pla_monomer, head_tag=">", tail_tag="<", separation=0.145)
copolymer.add_monomer(peg_monomer, head_tag=">", tail_tag="<", separation=0.145)
copolymer.add_monomer(pga_monomer, head_tag=">", tail_tag="<", separation=0.145)
copolymer.build_from_path(path, sequence="ABC")

hoomd_sim = HoomdSimulation(compound=copolymer, forcefield=gaff, r_cut=1.2)
hoomd_cap_displacement(
    n_steps=1000,
    compound=copolymer,
    sim=hoomd_sim,
    forces_handler=ForcesHandler(),
    max_displacement=1e-3,
    dt=1
)

hoomd_fire(compound=copolymer, n_iterations=5, n_steps=1000, sim=hoomd_sim, forces_handler=ForcesHandler())
copolymer.visualize()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.